<a href="https://colab.research.google.com/github/kumakun2008/CIFAR-10_Comp/blob/main/kumakun2008_ConvNeXt_1116.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CIFAR-10 画像分類（ImageNet-22K事前学習済み・99%超え仕様：ConvNeXt-Base + Swin-Base + ViT-Base/16）

## 工夫した点（箇条書き）

<table>
  <tr>
    <td><img src="https://github.com/kumakun2008/CIFAR-10_Comp/blob/main/assets/gemini_icon.png?raw=1" width="84" alt="生産者: Gemin 4 Hydrogen" /></td>
    <td>
      <strong>【生産者表示】Gemin 4 Hydrogen が作りました！！！</strong><br/>
      （顔写真付き・CIFAR-10高効率学習チューニング済み）
    </td>
  </tr>
</table>

* **生産者**：**Gemin 4 Hydrogenが作りました！！！**（上記の顔写真付き）
* **ImageNet（ImageNet-22K / 21K）で事前学習したモデルの重みを使用した**（`timm` の `convnext_base.fb_in22k_ft_in1k`, `swin_base_patch4_window7_224.ms_in22k_ft_in1k`, `vit_base_patch16_224.augreg_in21k_ft_in1k` を利用し、出力層を10クラスに差し替えて本ノートブック内でCIFAR-10全層ファインチューニングを実施）。
* **系統の異なる3アーキテクチャの選定**：
  * モダンCNNの **ConvNeXt-Base**、階層型Transformerの **Swin-Base**、グローバルAttention型Transformerの **ViT-Base/16** という特徴抽出の仕組みが異なる3モデルを採用した。
* **前処理・ルール準拠の正規化とGPU高速データ拡張**：
  * 前処理の正規化統計（`means`, `stds`）は配布コードのとおり **CIFAR-10訓練データから計算した値** をそのまま使用した。
  * 32×32画像をVRAMへ常駐させ、GPU上でバッチ一括リサイズ（`bicubic`, `224×224`）とデータ拡張（`RandomCrop`・`RandomHorizontalFlip`・明るさ/コントラスト揺らぎ・**Cutout**・**Mixup**）を実行した。
* **終盤のMixup Cooldownとアーキテクチャ別最適学習率**：
  * 配布コードのループ内 `optimizer` 再初期化を修正し、**AdamW** ＋ **Linear Warmup付きCosineAnnealing** ＋ **勾配クリッピング（max_norm=1.0）** を導入した。
  * モデルごとに最適な学習率を設定し、**終盤3エポックはMixup・Cutout・色揺らぎをオフにするCooldown学習**を行うことで、クリーンなCIFAR-10画像への精密な仕上げを行った。
* **検証（Validation）に基づく確認と仕上げ（テストデータは学習・選定に不使用）**：
  * まず訓練45,000枚 / 検証5,000枚に分割し、`Val. Acc`・検証データの混同行列・誤分類可視化で妥当性を確認した。
  * 設定確定後、訓練＋検証を合わせた **50,000枚全量** で3モデルをそれぞれファインチューニングした。
  * 最終テスト評価では、各モデル単体の `Test Acc` に加え、**4-View TTA（全体＋拡大ズーム × 左右反転の4視点平均）と3モデルのアンサンブル**を適用した。


In [ ]:
import os
import sys
import math
import copy
import time
import random
import shutil
import subprocess
import urllib.request

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision.datasets as datasets

try:
    import timm
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm'])
    import timm

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tqdm.notebook import trange
import matplotlib.pyplot as plt
import numpy as np

print(f'PyTorch: {torch.__version__} | timm: {timm.__version__}')


In [ ]:
SEED = 1234

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)


# CIFAR-10 データの読み込みと前処理（GPU常駐＋GPU一括リサイズ高速化）

In [ ]:
ROOT = '.data'
os.makedirs(ROOT, exist_ok=True)
tar_path = os.path.join(ROOT, 'cifar-10-python.tar.gz')
drive_tar_path = '/content/drive/MyDrive/Colab_Data/CIFAR10/cifar-10-python.tar.gz'
mirror_url = 'https://data.brainchip.com/dataset-mirror/cifar10/cifar-10-python.tar.gz'

if not os.path.exists(os.path.join(ROOT, 'cifar-10-batches-py')) and (
    not os.path.exists(tar_path) or os.path.getsize(tar_path) != 170498071
):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        if os.path.exists(drive_tar_path):
            print(f'Copying CIFAR-10 from Google Drive: {drive_tar_path}')
            shutil.copy2(drive_tar_path, tar_path)
    except Exception as e:
        print(f'Google Drive copy skipped ({e})')

    if not os.path.exists(tar_path) or os.path.getsize(tar_path) != 170498071:
        print(f'Downloading CIFAR-10 from mirror: {mirror_url}')
        urllib.request.urlretrieve(mirror_url, tar_path)

train_raw = datasets.CIFAR10(root=ROOT, train=True, download=True)
test_raw = datasets.CIFAR10(root=ROOT, train=False, download=True)

means = train_raw.data.mean(axis=(0, 1, 2)) / 255
stds = train_raw.data.std(axis=(0, 1, 2)) / 255

print(f'Calculated means: {means}')
print(f'Calculated stds: {stds}')


In [ ]:
IMAGE_SIZE = 224
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = (device.type == 'cuda')
amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
use_scaler = (use_amp and amp_dtype == torch.float16)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3) if device.type == 'cuda' else 0.0

BATCH_SIZE = 128 if vram_gb >= 22 else 64
EVAL_BATCH_SIZE = 100
LR_SCALE = BATCH_SIZE / 128.0

mean_t = torch.tensor(means, dtype=torch.float32).view(1, 3, 1, 1)
std_t = torch.tensor(stds, dtype=torch.float32).view(1, 3, 1, 1)

X_all_cpu = torch.from_numpy(train_raw.data).permute(0, 3, 1, 2).float().div_(255.0)
X_all_cpu = (X_all_cpu - mean_t) / std_t
Y_all_cpu = torch.tensor(train_raw.targets, dtype=torch.long)

X_test_cpu = torch.from_numpy(test_raw.data).permute(0, 3, 1, 2).float().div_(255.0)
X_test_cpu = (X_test_cpu - mean_t) / std_t
Y_test_cpu = torch.tensor(test_raw.targets, dtype=torch.long)

VALID_RATIO = 0.9
n_train_examples = int(len(X_all_cpu) * VALID_RATIO)

split_generator = torch.Generator().manual_seed(SEED)
perm_idx = torch.randperm(len(X_all_cpu), generator=split_generator)
train_idx = perm_idx[:n_train_examples]
valid_idx = perm_idx[n_train_examples:]

X_full = X_all_cpu.to(device)
Y_full = Y_all_cpu.to(device)
X_train = X_full[train_idx]
Y_train = Y_full[train_idx]
X_valid = X_full[valid_idx]
Y_valid = Y_full[valid_idx]
X_test = X_test_cpu.to(device)
Y_test = Y_test_cpu.to(device)

classes = test_raw.classes

print(f'Device: {device} (VRAM: {vram_gb:.1f} GB) | AMP: {use_amp} ({amp_dtype}) | BATCH_SIZE: {BATCH_SIZE}')
print(f'Train: {len(X_train)} | Valid: {len(X_valid)} | Full: {len(X_full)} | Test: {len(X_test)}')


In [ ]:
def normalize_image(image):
    image = image.clone()
    image_min = image.min()
    image_max = image.max()
    image.clamp_(min=image_min, max=image_max)
    image.add_(-image_min).div_(image_max - image_min + 1e-5)
    return image

def plot_images(images, labels, classes, normalize=False):
    n_images = len(images)
    rows = int(np.sqrt(n_images))
    cols = int(np.sqrt(n_images))
    fig = plt.figure(figsize=(10, 10))
    for i in range(rows * cols):
        ax = fig.add_subplot(rows, cols, i + 1)
        image = images[i]
        if normalize:
            image = normalize_image(image)
        ax.imshow(image.permute(1, 2, 0).cpu().numpy())
        ax.set_title(classes[labels[i]])
        ax.axis('off')

N_IMAGES = 25
plot_images(X_train[:N_IMAGES], Y_train[:N_IMAGES].tolist(), classes, normalize=True)


# モデル設計（ImageNet-22K / 21K 事前学習済み Baseモデル × 3系統）
1. **ConvNeXt-Base** (`convnext_base.fb_in22k_ft_in1k`)
2. **Swin-Base** (`swin_base_patch4_window7_224.ms_in22k_ft_in1k`)
3. **ViT-Base/16** (`vit_base_patch16_224.augreg_in21k_ft_in1k`)


In [ ]:
OUTPUT_DIM = 10

def build_convnext_base(num_classes=10):
    return timm.create_model('convnext_base.fb_in22k_ft_in1k', pretrained=True, num_classes=num_classes)

def build_swin_base(num_classes=10):
    return timm.create_model('swin_base_patch4_window7_224.ms_in22k_ft_in1k', pretrained=True, num_classes=num_classes)

def build_vit_base(num_classes=10):
    return timm.create_model('vit_base_patch16_224.augreg_in21k_ft_in1k', pretrained=True, num_classes=num_classes)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print('ImageNet-22K/21K model builders ready.')


# GPU一括リサイズ・Cooldown対応学習・4-View TTA評価関数

In [ ]:
def calculate_accuracy(y_pred, y):
    top_pred = y_pred.argmax(1, keepdim=True)
    correct = top_pred.eq(y.view_as(top_pred)).sum()
    acc = correct.float() / y.shape[0]
    return acc


def augment_and_resize_batch(x, image_size=224, pad=4, jitter_prob=0.5, erase_prob=0.25):
    B, C, H, W = x.shape
    x_pad = F.pad(x, (pad, pad, pad, pad), mode='reflect')
    top = torch.randint(0, 2 * pad + 1, (B, 1, 1, 1), device=x.device)
    left = torch.randint(0, 2 * pad + 1, (B, 1, 1, 1), device=x.device)
    rows = top + torch.arange(H, device=x.device).view(1, 1, H, 1)
    cols = left + torch.arange(W, device=x.device).view(1, 1, 1, W)
    b_idx = torch.arange(B, device=x.device).view(B, 1, 1, 1)
    c_idx = torch.arange(C, device=x.device).view(1, C, 1, 1)
    out = x_pad[b_idx, c_idx, rows, cols]

    flip_mask = (torch.rand(B, 1, 1, 1, device=x.device) < 0.5)
    out = torch.where(flip_mask, torch.flip(out, dims=[3]), out)

    if jitter_prob > 0:
        apply_jitter = (torch.rand(B, 1, 1, 1, device=x.device) < jitter_prob)
        scale = torch.empty(B, 1, 1, 1, device=x.device).uniform_(0.85, 1.15)
        shift = torch.empty(B, 1, 1, 1, device=x.device).uniform_(-0.15, 0.15)
        out = torch.where(apply_jitter, out * scale + shift, out)

    out = F.interpolate(out, size=(image_size, image_size), mode='bicubic', align_corners=False)

    if erase_prob > 0:
        cy = torch.randint(0, image_size, (B, 1, 1, 1), device=x.device)
        cx = torch.randint(0, image_size, (B, 1, 1, 1), device=x.device)
        half_h = torch.randint(16, 48, (B, 1, 1, 1), device=x.device)
        half_w = torch.randint(16, 48, (B, 1, 1, 1), device=x.device)
        apply_cut = (torch.rand(B, 1, 1, 1, device=x.device) < erase_prob)
        yy = torch.arange(image_size, device=x.device).view(1, 1, image_size, 1)
        xx = torch.arange(image_size, device=x.device).view(1, 1, 1, image_size)
        erase_mask = (
            apply_cut
            & (yy >= cy - half_h) & (yy < cy + half_h)
            & (xx >= cx - half_w) & (xx < cx + half_w)
        )
        out = out.masked_fill(erase_mask, 0.0)
    return out


def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1.0 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1.0 - lam) * criterion(pred, y_b)


def create_warmup_cosine_scheduler(optimizer, epochs, warmup_epochs=2, min_lr_ratio=0.01):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return float(epoch + 1) / float(max(1, warmup_epochs))
        progress = float(epoch - warmup_epochs) / float(max(1, epochs - warmup_epochs))
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        return min_lr_ratio + (1.0 - min_lr_ratio) * cosine
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def train_epoch_gpu(model, X_data, Y_data, batch_size, optimizer, criterion, scaler,
                    mixup_alpha=0.2, jitter_prob=0.5, erase_prob=0.25,
                    use_amp=True, amp_dtype=torch.float16, max_grad_norm=1.0):
    model.train()
    n = X_data.size(0)
    perm = torch.randperm(n, device=X_data.device)
    total_loss = 0.0
    total_acc = 0.0
    num_batches = 0

    for start in range(0, n, batch_size):
        idx = perm[start:start + batch_size]
        x = augment_and_resize_batch(
            X_data[idx], image_size=IMAGE_SIZE, jitter_prob=jitter_prob, erase_prob=erase_prob
        )
        y = Y_data[idx]

        optimizer.zero_grad(set_to_none=True)

        if mixup_alpha > 0:
            x_mixed, y_a, y_b, lam = mixup_data(x, y, alpha=mixup_alpha)
            with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=use_amp):
                y_pred = model(x_mixed)
                loss = mixup_criterion(criterion, y_pred, y_a, y_b, lam)
            acc = lam * calculate_accuracy(y_pred, y_a) + (1.0 - lam) * calculate_accuracy(y_pred, y_b)
        else:
            with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=use_amp):
                y_pred = model(x)
                loss = criterion(y_pred, y)
            acc = calculate_accuracy(y_pred, y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        if max_grad_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        total_acc += acc.item()
        num_batches += 1

    return total_loss / num_batches, total_acc / num_batches


def evaluate_gpu(model, X_data, Y_data, batch_size, criterion, use_amp=True, amp_dtype=torch.float16):
    model.eval()
    n = X_data.size(0)
    total_loss = 0.0
    total_acc = 0.0
    num_batches = 0

    with torch.no_grad():
        for start in range(0, n, batch_size):
            x = F.interpolate(X_data[start:start + batch_size], size=(IMAGE_SIZE, IMAGE_SIZE),
                              mode='bicubic', align_corners=False)
            y = Y_data[start:start + batch_size]
            with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=use_amp):
                y_pred = model(x)
                loss = criterion(y_pred, y)
            acc = calculate_accuracy(y_pred, y)
            total_loss += loss.item()
            total_acc += acc.item()
            num_batches += 1

    return total_loss / num_batches, total_acc / num_batches


def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs


# Phase 1：検証データ（Val. Acc）を用いたファインチューニング設定の確認
まずは `ConvNeXt-Base（ImageNet-22K）` を使って、訓練45,000枚 / 検証5,000枚で学習曲線と `Val. Acc` を確認します（終盤3エポックはCooldown、テストデータは一切参照しません）。


In [ ]:
EPOCHS = 14
COOLDOWN_EPOCHS = 3
WEIGHT_DECAY = 1e-2
MIXUP_ALPHA = 0.2
LABEL_SMOOTHING = 0.05

set_seed(SEED)
val_model = build_convnext_base(OUTPUT_DIM).to(device)
print(f'ConvNeXt-Base trainable parameters: {count_parameters(val_model):,}')

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING).to(device)
clean_criterion = nn.CrossEntropyLoss(label_smoothing=0.02).to(device)
eval_criterion = nn.CrossEntropyLoss().to(device)

optimizer = optim.AdamW(val_model.parameters(), lr=1.0e-4 * LR_SCALE, weight_decay=WEIGHT_DECAY)
scheduler = create_warmup_cosine_scheduler(optimizer, epochs=EPOCHS, warmup_epochs=2)
scaler = torch.amp.GradScaler('cuda', enabled=use_scaler)

best_valid_acc = 0.0
best_epoch = 0
history = {'train_loss': [], 'train_acc': [], 'valid_loss': [], 'valid_acc': []}

for epoch in trange(EPOCHS, desc="Phase 1 (Val Check)"):
    start_time = time.monotonic()
    in_cooldown = (epoch >= EPOCHS - COOLDOWN_EPOCHS)

    train_loss, train_acc = train_epoch_gpu(
        val_model, X_train, Y_train, BATCH_SIZE, optimizer,
        clean_criterion if in_cooldown else criterion, scaler,
        mixup_alpha=0.0 if in_cooldown else MIXUP_ALPHA,
        jitter_prob=0.0 if in_cooldown else 0.5,
        erase_prob=0.0 if in_cooldown else 0.25,
        use_amp=use_amp, amp_dtype=amp_dtype
    )
    valid_loss, valid_acc = evaluate_gpu(
        val_model, X_valid, Y_valid, EVAL_BATCH_SIZE, eval_criterion,
        use_amp=use_amp, amp_dtype=amp_dtype
    )
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['valid_loss'].append(valid_loss)
    history['valid_acc'].append(valid_acc)

    if valid_acc > best_valid_acc:
        best_valid_acc = valid_acc
        best_epoch = epoch + 1
        torch.save(val_model.state_dict(), 'convnext_base_val_best.pt')

    end_time = time.monotonic()
    epoch_mins, epoch_secs = epoch_time(start_time, end_time)
    tag = ' [Cooldown]' if in_cooldown else ''

    print(f'Epoch: {epoch+1:02}{tag} | Time: {epoch_mins}m {epoch_secs}s | LR: {scheduler.get_last_lr()[0]:.6f}')
    print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. Acc: {valid_acc*100:.2f}%')

print(f'\nBest Val. Acc: {best_valid_acc*100:.2f}% (at Epoch {best_epoch})')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['valid_loss'], label='Valid Loss')
axes[0].set_title('Loss Curve (Phase 1)')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot([a * 100 for a in history['train_acc']], label='Train Acc')
axes[1].plot([a * 100 for a in history['valid_acc']], label='Valid Acc')
axes[1].set_title('Accuracy Curve (Phase 1)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
plt.show()


## 検証データ（Validation）での誤差分析（混同行列・誤分類画像）
テストデータを分析に使うのではなく、検証データで弱点を確認します。


In [ ]:
def predict_4view_tta_probs(models_list, x_32, image_size=224, use_amp=True, amp_dtype=torch.float16):
    # 4 views: full, full+hflip, center-zoom(28->224), center-zoom+hflip
    x_full = F.interpolate(x_32, size=(image_size, image_size), mode='bicubic', align_corners=False)
    x_zoom = F.interpolate(x_32[:, :, 2:30, 2:30], size=(image_size, image_size), mode='bicubic', align_corners=False)
    views = [
        x_full,
        torch.flip(x_full, dims=[3]),
        x_zoom,
        torch.flip(x_zoom, dims=[3]),
    ]
    probs_sum = 0.0
    with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=use_amp):
        for m in models_list:
            m_probs = 0.0
            for v in views:
                m_probs = m_probs + F.softmax(m(v).float(), dim=-1)
            probs_sum = probs_sum + (m_probs / len(views))
    return probs_sum / len(models_list)


def get_predictions_gpu(model, X_data, Y_data, batch_size, use_tta=True, use_amp=True, amp_dtype=torch.float16):
    model.eval()
    probs = []
    with torch.no_grad():
        for start in range(0, X_data.size(0), batch_size):
            x_32 = X_data[start:start + batch_size]
            if use_tta:
                y_prob = predict_4view_tta_probs([model], x_32, IMAGE_SIZE, use_amp=use_amp, amp_dtype=amp_dtype)
            else:
                x_224 = F.interpolate(x_32, size=(IMAGE_SIZE, IMAGE_SIZE), mode='bicubic', align_corners=False)
                with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=use_amp):
                    y_prob = F.softmax(model(x_224).float(), dim=-1)
            probs.append(y_prob.cpu())
    return X_data.cpu(), Y_data.cpu(), torch.cat(probs, dim=0)


def plot_confusion_matrix(labels, pred_labels, classes):
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(1, 1, 1)
    cm = confusion_matrix(labels, pred_labels)
    cm = ConfusionMatrixDisplay(cm, display_labels=classes)
    cm.plot(values_format='d', cmap='Blues', ax=ax)
    plt.xticks(rotation=20)


def plot_most_incorrect(incorrect, classes, n_images, normalize=True):
    rows = int(np.sqrt(n_images))
    cols = int(np.sqrt(n_images))
    fig = plt.figure(figsize=(20, 16))
    for i in range(min(rows * cols, len(incorrect))):
        ax = fig.add_subplot(rows, cols, i + 1)
        image, true_label, probs = incorrect[i]
        image = image.permute(1, 2, 0)
        true_prob = probs[true_label]
        incorrect_prob, incorrect_label = torch.max(probs, dim=0)
        true_class = classes[true_label]
        incorrect_class = classes[incorrect_label]
        if normalize:
            image = normalize_image(image)
        ax.imshow(image.cpu().numpy())
        ax.set_title(f'true: {true_class} ({true_prob:.3f})\npred: {incorrect_class} ({incorrect_prob:.3f})')
        ax.axis('off')
    fig.subplots_adjust(hspace=0.4)


val_model.load_state_dict(torch.load('convnext_base_val_best.pt', map_location=device))
val_images, val_labels, val_probs = get_predictions_gpu(
    val_model, X_valid, Y_valid, EVAL_BATCH_SIZE, use_tta=True, use_amp=use_amp, amp_dtype=amp_dtype
)
val_preds = torch.argmax(val_probs, dim=1)
print(f'Validation Acc (with 4-View TTA): {(val_preds == val_labels).float().mean().item()*100:.2f}%')
plot_confusion_matrix(val_labels, val_preds, classes)


In [ ]:
corrects = torch.eq(val_labels, val_preds)
incorrect_examples = []
for image, label, prob, correct in zip(val_images, val_labels, val_probs, corrects):
    if not correct:
        incorrect_examples.append((image, label, prob))

incorrect_examples.sort(reverse=True, key=lambda x: torch.max(x[2], dim=0).values)
plot_most_incorrect(incorrect_examples, classes, 25)


# Phase 2：訓練＋検証の全50,000枚で3モデルをファインチューニング（アンサンブル用）
Phase 1 の設定（終盤3エポックのCooldown含む）をもとに、訓練＋検証の **50,000枚すべて** で系統の異なる3つのImageNet-22K/21K事前学習モデルを学習します。
1. **ConvNeXt-Base**（`lr=1.0e-4`）
2. **Swin-Base**（`lr=6.0e-5`）
3. **ViT-Base/16**（`lr=4.0e-5`）


In [ ]:
del val_model
if device.type == 'cuda':
    torch.cuda.empty_cache()

def train_full_dataset(model_fn, model_name, save_path, lr=1e-4, epochs=14, cooldown_epochs=3, seed=1234):
    set_seed(seed)
    model = model_fn().to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING).to(device)
    clean_criterion = nn.CrossEntropyLoss(label_smoothing=0.02).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = create_warmup_cosine_scheduler(optimizer, epochs=epochs, warmup_epochs=2)
    scaler = torch.amp.GradScaler('cuda', enabled=use_scaler)

    print(f'=== Start Full Fine-tuning: {model_name} ({count_parameters(model):,} params, lr={lr:.1e}) ===')
    for epoch in trange(epochs, desc=model_name):
        start_time = time.monotonic()
        in_cooldown = (epoch >= epochs - cooldown_epochs)
        train_loss, train_acc = train_epoch_gpu(
            model, X_full, Y_full, BATCH_SIZE, optimizer,
            clean_criterion if in_cooldown else criterion, scaler,
            mixup_alpha=0.0 if in_cooldown else MIXUP_ALPHA,
            jitter_prob=0.0 if in_cooldown else 0.5,
            erase_prob=0.0 if in_cooldown else 0.25,
            use_amp=use_amp, amp_dtype=amp_dtype
        )
        scheduler.step()
        end_time = time.monotonic()
        epoch_mins, epoch_secs = epoch_time(start_time, end_time)

        if (epoch + 1) % 4 == 0 or epoch == 0 or in_cooldown:
            tag = ' [Cooldown]' if in_cooldown else ''
            print(f'[{model_name}] Epoch {epoch+1:02}/{epochs}{tag} ({epoch_mins}m {epoch_secs}s) | '
                  f'Train Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')

    torch.save(model.state_dict(), save_path)
    print(f'Saved {model_name} weights to {save_path}\n')
    model.cpu()
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return model


In [ ]:
FULL_EPOCHS = 14
COOLDOWN_EPOCHS = 3

model_convnext = train_full_dataset(
    lambda: build_convnext_base(OUTPUT_DIM),
    model_name='ConvNeXt-Base (IN22K)',
    save_path='convnext_base_full.pt',
    lr=1.0e-4 * LR_SCALE,
    epochs=FULL_EPOCHS,
    cooldown_epochs=COOLDOWN_EPOCHS,
    seed=SEED
)

model_swin = train_full_dataset(
    lambda: build_swin_base(OUTPUT_DIM),
    model_name='Swin-Base (IN22K)',
    save_path='swin_base_full.pt',
    lr=6.0e-5 * LR_SCALE,
    epochs=FULL_EPOCHS,
    cooldown_epochs=COOLDOWN_EPOCHS,
    seed=SEED + 1
)

model_vit = train_full_dataset(
    lambda: build_vit_base(OUTPUT_DIM),
    model_name='ViT-Base/16 (IN21K)',
    save_path='vit_base_full.pt',
    lr=4.0e-5 * LR_SCALE,
    epochs=FULL_EPOCHS,
    cooldown_epochs=COOLDOWN_EPOCHS,
    seed=SEED + 2
)


# 最終テスト評価（10,000枚）

In [ ]:
# 各モデル単体のテスト精度（配布コード準拠の評価）
model_convnext.load_state_dict(torch.load('convnext_base_full.pt', map_location=device))
model_swin.load_state_dict(torch.load('swin_base_full.pt', map_location=device))
model_vit.load_state_dict(torch.load('vit_base_full.pt', map_location=device))

model_convnext.to(device)
model_swin.to(device)
model_vit.to(device)

for name, m in [('ConvNeXt-Base', model_convnext),
                ('Swin-Base', model_swin),
                ('ViT-Base/16', model_vit)]:
    test_loss, test_acc = evaluate_gpu(
        m, X_test, Y_test, EVAL_BATCH_SIZE, eval_criterion, use_amp=use_amp, amp_dtype=amp_dtype
    )
    print(f'[{name}] Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}%')


In [ ]:
# 3モデル アンサンブル + 4-View TTAによる最終テスト評価
def evaluate_ensemble_tta_gpu(models_list, X_data, Y_data, batch_size, use_amp=True, amp_dtype=torch.float16):
    for m in models_list:
        m.eval()

    n = X_data.size(0)
    total_loss = 0.0
    total_acc = 0.0
    num_batches = 0

    with torch.no_grad():
        for start in range(0, n, batch_size):
            x_32 = X_data[start:start + batch_size]
            y = Y_data[start:start + batch_size]

            avg_probs = predict_4view_tta_probs(
                models_list, x_32, IMAGE_SIZE, use_amp=use_amp, amp_dtype=amp_dtype
            )
            loss = F.nll_loss(torch.log(avg_probs.clamp(min=1e-12)), y)
            acc = calculate_accuracy(avg_probs, y)

            total_loss += loss.item()
            total_acc += acc.item()
            num_batches += 1

    return total_loss / num_batches, total_acc / num_batches


ensemble_models = [model_convnext, model_swin, model_vit]
test_loss, test_acc = evaluate_ensemble_tta_gpu(
    ensemble_models, X_test, Y_test, EVAL_BATCH_SIZE, use_amp=use_amp, amp_dtype=amp_dtype
)

print(f'Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}%')
